In [14]:
import pandas as pd
import geopandas as gpd
collisions = pd.read_csv("data/raw/dft-road-casualty-statistics-collision-last-5-years.csv", low_memory=False)
casualties = pd.read_csv("data/raw/dft-road-casualty-statistics-casualty-last-5-years.csv", low_memory=False)
collisions.head() # what is one row?
collisions.info() # column types and blanks
collisions.describe() # impossible numbers
collisions.isna().sum() # blanks per column
collisions.duplicated().sum() # repeated rows

<class 'pandas.DataFrame'>
RangeIndex: 513801 entries, 0 to 513800
Data columns (total 44 columns):
 #   Column                                            Non-Null Count   Dtype  
---  ------                                            --------------   -----  
 0   collision_index                                   513801 non-null  str    
 1   collision_year                                    513801 non-null  int64  
 2   collision_ref_no                                  513801 non-null  str    
 3   location_easting_osgr                             513748 non-null  float64
 4   location_northing_osgr                            513748 non-null  float64
 5   longitude                                         513748 non-null  float64
 6   latitude                                          513748 non-null  float64
 7   police_force                                      513801 non-null  int64  
 8   collision_severity                                513801 non-null  int64  
 9   number_of_vehic

np.int64(0)

In [16]:
import geopandas as gpd

# crs= tells GeoPandas what coordinate system the input numbers are already in. 
# .to_crs(27700) transforms those numbers into British National Grid eastings and northings.

# Create points from the collision coordinates.
# Pass easting first and northing second.
collision_points = gpd.GeoDataFrame(
    collisions,
    geometry=gpd.points_from_xy(
        collisions["location_easting_osgr"],
        collisions["location_northing_osgr"],
    ),
    crs="EPSG:27700",
)


# Use the same CRS for the boundary and the points before filtering. 
# This keeps collisions strictly inside the buffered boundary.
boundary = (
    gpd.read_file("data/clean/london_plus_400m.geojson")
    .to_crs(27700)
    .geometry.iloc[0]
)


# Filter the collisions to those strictly inside the buffered boundary.
collisions = collisions.loc[collision_points.within(boundary)].copy()


#### Filtering casualties to pedestrians aged 4 to 11 

In [ ]:
is_pedestrian = casualties["casualty_class"].eq(3)
is_aged_4_to_11 = casualties["age_of_casualty"].between(4, 11)

child_pedestrians = casualties.loc[is_pedestrian & is_aged_4_to_11].copy()

print(f"Casualties kept: {len(child_pedestrians):,}")
print(child_pedestrians["age_of_casualty"].value_counts().sort_index())

london_child_pedestrians = child_pedestrians.loc[
    child_pedestrians["collision_index"].astype(str).isin(
        collisions["collision_index"].astype(str)
    )
].copy()

print(f"Casualties within the boundary: {len(london_child_pedestrians):,}")

# Checking how many rows were removed:
rows_removed = len(casualties) - len(child_pedestrians)

print(f"Original casualty rows: {len(casualties):,}")
print(f"Rows kept: {len(child_pedestrians):,}")
print(f"Rows removed: {rows_removed:,}")

Casualties kept: 11,167
age_of_casualty
4      974
5     1005
6     1047
7     1014
8     1052
9     1211
10    1669
11    3195
Name: count, dtype: int64
Casualties within the boundary: 1,890
Original casualty rows: 652,821
Rows kept: 11,167
Rows removed: 641,654
